# 08 · Head refinement by analysis-by-synthesis

The head fit (`fitting/head.py`) only matches landmarks and the outline; the texture is projected
afterwards and never corrects the shape. Here the ICT-FaceKit head is **rendered** (nvdiffrast)
with a texture and a light, compared with every pixel of both head videos, and everything is
nudged to make the renders match the frames:

| What | Shared or per frame | Phase |
| --- | --- | --- |
| Skin colour (albedo), a texture pyramid | shared | 1 → |
| Light: 2nd-order spherical harmonics, RGB | per video | 1 → |
| Head pose, expression weights, gaze | per frame | 2 → |
| Identity (ICT's 100 shapes) | shared | 2 → |
| Smooth per-vertex shape changes (broad and fine), where the teeth sit | shared | 3 → |
| Her own version of each of the 53 blendshapes (broad and fine) | shared | 4 → |
| Wrinkle maps: colour changes per blendshape on the face | shared | 5 |

Every 8th frame is **held out**: it gets its own pose, face and gaze fitted, but nothing shared
learns from it, so its scores say how well the head generalises.

What the first run taught:

- The teeth, gums and tongue followed the nearest skin vertex's shape change (2 cm away on
  average, 494 different ones), so reshaping the lips bent the teeth. They are rigid now: the
  upper set with the skull, the lower set with ICT's jaw blendshapes, each with one learned shift.
- Only the head's outline was antialiased, so the contours inside it (lips against teeth, lids
  against eyes) gave the shape no gradient; the lips moved by landmarks alone. The colour is
  antialiased now.
- One smoothing scale (about a centimetre) cannot shape a lip; a second, fine layer can, at a
  higher price.
- The neutral turntable's frames leaned on seven blendshapes (one at 0.98) and their
  corrections, so the face with every weight at zero, the avatar's rest face, was never fitted
  and grimaced. The turntable now only blinks and looks around.
- One texture for all frames bakes the forehead's wrinkles in, half-strength, everywhere. Wrinkle
  maps add colour per blendshape, as games do (MetaHuman's animated maps).

Run `go-task fit` first (this starts from its results). Everything is written to
`outputs/head_refinement/`; nothing else in the pipeline reads it yet.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import nvdiffrast.torch as dr
import roma
import torch
import torch.nn.functional as F

from avatar_splats.body_models import ict
from avatar_splats.body_models.ict import IctFaceKit
from avatar_splats.config import load_config
from avatar_splats.data.video import read_frames
from avatar_splats.faces.dense import FACE_OVAL, NUM_MESH_LANDMARKS, landmark_vertices
from avatar_splats.faces.pipeline import NEUTRAL_VIDEO
from avatar_splats.fitting import HEAD_VIDEOS, HeadFit, fitted_path
from avatar_splats.fitting.head import CHIN, ICT_TO_WORLD, facing, landmark_targets
from avatar_splats.fitting.losses import keypoint_loss, second_difference
from avatar_splats.preprocess import Preprocessed, preprocessed_path

CONFIG = "../configs/default.yaml"  # the capture: e.g. "../configs/performer-2.yaml"
config = load_config(CONFIG)
device = "cuda"
out_dir = config.outputs / "head_refinement"
out_dir.mkdir(parents=True, exist_ok=True)

## Settings

Lower `RESOLUTION` or `CHUNK` if the GPU runs out of memory. `ITERATIONS` per phase; a phase set
to 0 is skipped.

In [ ]:
RESOLUTION = 512  # pixels per side the frames are compared at
TEXTURE_SIZE = 512  # texels per side of each of the atlas's tiles
PYRAMID_LEVELS = 6  # texture resolutions learned together: 16, 32, ... TEXTURE_SIZE
HOLD_OUT = 8  # every HOLD_OUT-th frame is held out
CHUNK = 27  # frames rendered at once
SMOOTHING_STEPS = 40  # neighbour averages applied to broad shape changes: more is smoother
FINE_SMOOTHING_STEPS = 4  # the same for the fine layer, a few millimetres across
WRINKLE_SIZE = 256  # texels per side of a wrinkle map, over the face's tile
EDGE_BAND = 4  # pixels along the mask's edge left out of the colour comparison
NEAR, FAR = 0.05, 5.0  # metres

ITERATIONS = {"appearance": 300, "face": 600, "shape": 600, "expressions": 600, "wrinkles": 400}

LEARNING_RATES = {
    "texture": 0.01,
    "light": 0.01,
    "pose": 0.001,
    "expressions": 0.02,
    "gaze": 0.005,
    "identity": 0.01,
    "offsets": 0.02,  # millimetres per step, before smoothing
    "correctives": 0.02,
    "mouth": 0.05,  # millimetres per step
    "wrinkles": 0.005,
}

WEIGHTS = {
    "photo": 1.0,  # mean absolute colour difference, colours in [0, 1]
    "silhouette": 1.0,  # mean squared difference above the chin
    "landmarks": 1.0,  # 468 MediaPipe mesh landmarks, in image heights (keypoint_loss)
    "irises": 1.0,
    "temporal": config.fitting.smoothness_weight,
    "identity_prior": config.fitting.identity_prior_weight,
    "sparsity": config.fitting.expression_sparsity_weight,
    "gaze_prior": config.fitting.gaze_prior_weight,
    "offsets": 1e-3,  # per mm² of broad shape change, averaged over vertices
    "fine_offsets": 1e-2,  # per mm² of fine shape change
    "correctives": 1e-3,  # per mm² of broad blendshape change, averaged over vertices and shapes
    "fine_correctives": 1e-2,
    "mouth": 1e-3,  # per mm² of the teeth's shifts
    "wrinkles": 0.05,  # per unit of mean absolute colour change: keeps the maps sparse
}
OVAL_WEIGHT = 0.3  # the face-outline landmarks slide over the head as it turns

## Data: frames, masks, landmarks and the current head fit

In [ ]:
model = IctFaceKit(config.body_models.ict_facekit).to(device)
NUM_VERTICES = model.neutral.shape[0]
SKIN_COUNT = ict.SKIN.stop


class Video:
    def __init__(self, name):
        self.name = name
        self.fit = HeadFit.load(fitted_path(config, name))
        pre = Preprocessed.load(preprocessed_path(config, name))
        width, height = (int(v) for v in self.fit.image_size)
        self.scale = RESOLUTION / max(width, height)
        self.camera = self.fit.camera().scaled(self.scale).to(device)
        size = (self.camera.height, self.camera.width)

        everything = list(read_frames(getattr(config.capture, name), step=1))
        frames = np.stack([everything[i] for i in pre.frame_indices])
        images = torch.from_numpy(frames).to(device).permute(0, 3, 1, 2).float() / 255
        self.images = F.interpolate(images, size, mode="area").permute(0, 2, 3, 1).contiguous()
        masks = torch.from_numpy(pre.masks).to(device).float()[:, None]
        self.masks = F.interpolate(masks, size, mode="area")[:, 0]
        hard = (self.masks > 0.5).float()[:, None]
        band = 2 * EDGE_BAND + 1
        self.inner = (-F.max_pool2d(-hard, band, stride=1, padding=EDGE_BAND))[:, 0]

        targets = landmark_targets(pre.face_landmarks, self.scale, model, device)
        self.landmarks = torch.nan_to_num(targets.mediapipe[:, :NUM_MESH_LANDMARKS, :2])
        weights = torch.ones(NUM_MESH_LANDMARKS, device=device)
        weights[list(FACE_OVAL)] = OVAL_WEIGHT
        self.landmark_weights = weights * targets.found[:, None]
        self.irises, self.found = targets.irises, targets.found
        self.face_landmarks = pre.face_landmarks

        count = len(frames)
        self.held_out = torch.zeros(count, dtype=torch.bool)
        self.held_out[HOLD_OUT // 2 :: HOLD_OUT] = True
        trained, held = torch.nonzero(~self.held_out)[:, 0], torch.nonzero(self.held_out)[:, 0]
        self.chunks = [(c, False) for c in trained.split(CHUNK)] + [
            (c, True) for c in held.split(CHUNK)
        ]

        fit = self.fit

        def p(a):
            return torch.nn.Parameter(torch.from_numpy(np.asarray(a)).float().to(device))

        self.rotations = p(fit.rotations)
        self.translations = p(fit.translations)
        weights = np.clip(fit.expressions, 1e-4, 1 - 1e-4)
        self.expression_logits = p(np.log(weights / (1 - weights)))
        self.eye_angles = p(fit.eye_rotations[..., :2])
        self.light = torch.nn.Parameter(torch.zeros(9, 3, device=device))
        with torch.no_grad():
            self.light[0] = 1 / 0.282095  # uniform light of strength 1
        # The turntable is neutral by definition: only blinks and gaze move its face, so its
        # frames, not blendshapes, decide the head's neutral shape and texture.
        moves = [
            name != NEUTRAL_VIDEO or expression.startswith(("eyeBlink", "eyeLook"))
            for expression in model.expression_names
        ]
        self.expression_mask = torch.tensor(moves, dtype=torch.float32, device=device)

    def expressions(self):
        return torch.sigmoid(self.expression_logits) * self.expression_mask

    def eye_rotations(self):
        return F.pad(self.eye_angles, (0, 1))

    def __len__(self):
        return len(self.images)


videos = [Video(name) for name in HEAD_VIDEOS]
centre = torch.from_numpy(videos[0].fit.centre).float().to(device)
identity = torch.nn.Parameter(torch.from_numpy(videos[0].fit.identity).float().to(device)[None])

dense_vertices = torch.from_numpy(
    landmark_vertices(
        model,
        [(v.fit, v.face_landmarks) for v in videos],
        config.faces.frontal_yaw,
        config.faces.correspondence_cap,
        device,
    )
).to(device)
pupils = model.pupils().to(device)
chin_vertex = int(model.landmarks[CHIN])
for v in videos:
    print(f"{v.name}: {len(v)} frames, {int(v.held_out.sum())} held out")

## The model

**Shape changes** are smooth by construction: the optimiser moves free values per skin vertex,
and the head gets them after rounds of averaging each vertex with its neighbours (the lessons
say free per-vertex offsets fold the surface). Two layers: a broad one (`SMOOTHING_STEPS`) and a
fine one (`FINE_SMOOTHING_STEPS`), which costs ten times as much per millimetre. The mouth and
eye sockets follow the nearest skin vertex; each eyeball moves whole, so it stays a sphere. The
teeth, gums and tongue get no shape change: the upper set (what `jawOpen` leaves in place) and
the lower set each get one learned shift. Her **blendshapes** get the same two layers, one pair
per blendshape.

**Rendering**: albedo = the texture plus, on the face's tile, each blendshape's wrinkle map times
its weight; colour = albedo × the light the surface normal receives (spherical harmonics). With
the light separate, the albedo should come out with less of the videos' shading baked in. The
colour is antialiased, so contours inside the head pull on its shape.

In [ ]:
skin_faces = model.faces[model.faces.max(dim=1).values < SKIN_COUNT]
edges = torch.cat([skin_faces[:, [0, 1]], skin_faces[:, [1, 2]], skin_faces[:, [2, 0]]])
edges = torch.cat(
    [edges, edges.flip(1), torch.arange(SKIN_COUNT, device=device)[:, None].repeat(1, 2)]
)
edges = torch.unique(edges, dim=0)
degree = torch.bincount(edges[:, 0], minlength=SKIN_COUNT).float()
averaging = torch.sparse_coo_tensor(
    edges.T, 1 / degree[edges[:, 0]], (SKIN_COUNT, SKIN_COUNT), check_invariants=True
).coalesce()


def smooth(values, steps):
    # (SKIN_COUNT, C) -> (SKIN_COUNT, C)
    for _ in range(steps):
        values = torch.sparse.mm(averaging, values)
    return values


SOCKETS_END = 14062  # mouth socket and eye sockets: joined to the skin at the lips and lids
INTERIOR = slice(SOCKETS_END, ict.EYEBALLS.start)  # gums and tongue, then the teeth
with torch.no_grad():
    rest = model.neutral[SKIN_COUNT:]
    nearest_skin = torch.cat(
        [torch.cdist(part, model.neutral[:SKIN_COUNT]).argmin(dim=1) for part in rest.split(4096)]
    )
    jaw = model.expression_basis[model.expression_names.index("jawOpen")].norm(dim=-1)
    lower_jaw = torch.zeros(NUM_VERTICES, 1, device=device)
    lower_jaw[INTERIOR] = (jaw[INTERIOR] > 1e-3).float()[:, None]
    upper_jaw = torch.zeros(NUM_VERTICES, 1, device=device)
    upper_jaw[INTERIOR] = 1 - lower_jaw[INTERIOR]


def spread(skin):
    # (..., SKIN_COUNT, 3) changes of the skin -> (..., V, 3) changes of every vertex
    batch = skin.shape[:-2]
    parts = [
        skin,
        skin[..., nearest_skin[: SOCKETS_END - SKIN_COUNT], :],
        skin.new_zeros(*batch, INTERIOR.stop - INTERIOR.start, 3),
    ]
    for eyeball in ict.EYEBALL_ORDER:
        followed = nearest_skin[eyeball.start - SKIN_COUNT : eyeball.stop - SKIN_COUNT]
        whole = skin[..., followed, :].mean(dim=-2, keepdim=True)
        parts.append(whole.expand(*batch, eyeball.stop - eyeball.start, 3))
    return torch.cat(parts, dim=-2)


MILLIMETRE = 1e-3
E3 = model.num_expressions * 3
offset_values = torch.nn.Parameter(torch.zeros(SKIN_COUNT, 3, device=device))
fine_offset_values = torch.nn.Parameter(torch.zeros(SKIN_COUNT, 3, device=device))
corrective_values = torch.nn.Parameter(torch.zeros(SKIN_COUNT, E3, device=device))
fine_corrective_values = torch.nn.Parameter(torch.zeros(SKIN_COUNT, E3, device=device))
mouth_shifts = torch.nn.Parameter(torch.zeros(2, 3, device=device))  # upper, lower; mm, ICT axes


def offset_layers():
    # broad and fine (SKIN_COUNT, 3) shape changes of the skin, in millimetres
    return smooth(offset_values, SMOOTHING_STEPS), smooth(fine_offset_values, FINE_SMOOTHING_STEPS)


def corrective_layers():
    # broad and fine (SKIN_COUNT, E * 3) blendshape changes of the skin, in millimetres
    return (
        smooth(corrective_values, SMOOTHING_STEPS),
        smooth(fine_corrective_values, FINE_SMOOTHING_STEPS),
    )


def offsets():
    # (V, 3) metres: the skin's shape changes and the teeth's shifts
    broad, fine = offset_layers()
    teeth = upper_jaw * mouth_shifts[0] + lower_jaw * mouth_shifts[1]
    return (spread(broad + fine) + teeth) * MILLIMETRE


def correctives():
    # (E, V, 3) metres
    broad, fine = corrective_layers()
    skin = (broad + fine).reshape(SKIN_COUNT, -1, 3).transpose(0, 1)
    return spread(skin) * MILLIMETRE


def posed_vertices(shared, expressions, eye_rotations, rotations, translations):
    # (B, V, 3) world vertices
    local = (
        model(shared["identity"], expressions)
        + shared["offsets"]
        + torch.einsum("be,evc->bvc", expressions, shared["correctives"])
    )
    local = ict.turn_eyes(local, eye_rotations) - centre
    rotation = roma.rotvec_to_rotmat(rotations) @ ICT_TO_WORLD.to(local)
    return local @ rotation.transpose(-1, -2) + translations[:, None]


def vertex_normals(vertices, faces):
    corners = vertices[:, faces]  # (B, F, 3, 3)
    face_normals = torch.linalg.cross(
        corners[:, :, 1] - corners[:, :, 0], corners[:, :, 2] - corners[:, :, 0]
    )
    normals = torch.zeros_like(vertices)
    for k in range(3):
        normals = normals.index_add(1, faces[:, k], face_normals)
    return F.normalize(normals, dim=-1)


def sh_basis(n):
    x, y, z = n.unbind(-1)
    return torch.stack(
        [
            0.282095 * torch.ones_like(x),
            0.488603 * y,
            0.488603 * z,
            0.488603 * x,
            1.092548 * x * y,
            1.092548 * y * z,
            0.315392 * (3 * z * z - 1),
            1.092548 * x * z,
            0.546274 * (x * x - y * y),
        ],
        dim=-1,
    )


def to_clip(vertices, camera):
    # world vertices -> nvdiffrast clip space; row 0 of the image is the top, as in the frames
    view = vertices @ camera.view[:3, :3].T + camera.view[:3, 3]
    x, y, z = view.unbind(-1)
    K, W, H = camera.K, camera.width, camera.height
    return torch.stack(
        [
            2 * K[0, 0] / W * x + (2 * K[0, 2] / W - 1) * z,
            2 * K[1, 1] / H * y + (2 * K[1, 2] / H - 1) * z,
            (FAR + NEAR) / (FAR - NEAR) * z - 2 * FAR * NEAR / (FAR - NEAR),
            z,
        ],
        dim=-1,
    ).contiguous()


tri = model.faces.int().contiguous()
corner_tri = torch.arange(3 * len(tri), device=device, dtype=torch.int32).reshape(-1, 3)
corner_uvs = model.uvs.reshape(-1, 2).clone()
corner_uvs[:, 1] = 1 - corner_uvs[:, 1]  # the texture's row 0 is its top, as in an image
corner_uvs = corner_uvs[None].contiguous()
context = dr.RasterizeCudaContext()

ATLAS = (TEXTURE_SIZE, ict.UV_TILES * TEXTURE_SIZE)
pyramid = torch.nn.ParameterList(
    [
        torch.nn.Parameter(torch.zeros(1, 3, ATLAS[0] >> k, ATLAS[1] >> k, device=device))
        for k in reversed(range(PYRAMID_LEVELS))
    ]
)
with torch.no_grad():  # start from the performer's mean colour
    v = videos[0]
    pyramid[0][:] = (
        (v.images * v.inner[..., None]).sum((0, 1, 2)).div(v.inner.sum())[None, :, None, None]
    )


def texture(levels=None):
    # (1, H, W, 3) albedo atlas
    levels = pyramid if levels is None else levels
    total = sum(
        F.interpolate(level, ATLAS, mode="bilinear", align_corners=False) for level in levels
    )
    return total.permute(0, 2, 3, 1).contiguous()


# Wrinkle maps: per blendshape, a colour change over the face's tile (u from 0 to 1 / UV_TILES).
wrinkle_maps = torch.nn.Parameter(
    torch.zeros(model.num_expressions, WRINKLE_SIZE, WRINKLE_SIZE, 3, device=device)
)


def wrinkles_of(expressions, maps):
    # (B, S, S, 3) each frame's colour change on the face, for (B, E) expression weights
    return torch.einsum("be,ehwc->bhwc", expressions, maps).contiguous()


def render(vertices, camera, albedo, light, wrinkles=None):
    # albedo: (1, H, W, 3) atlas; light: (9, 3); wrinkles: (B, S, S, 3) or None
    clip = to_clip(vertices, camera)
    rast, _ = dr.rasterize(context, clip, tri, (camera.height, camera.width))
    covered = (rast[..., 3] > 0).float()[..., None]
    uv, _ = dr.interpolate(corner_uvs, rast, corner_tri)
    colour = dr.texture(albedo, uv, filter_mode="linear")
    if wrinkles is not None:
        on_face = (uv[..., :1] < 1 / ict.UV_TILES).float()
        face_uv = torch.cat([uv[..., :1] * ict.UV_TILES, uv[..., 1:]], -1).contiguous()
        colour = colour + on_face * dr.texture(wrinkles, face_uv, filter_mode="linear")
    colour = colour.clamp(0, 1) * covered
    normals, _ = dr.interpolate(vertex_normals(vertices, model.faces).contiguous(), rast, tri)
    shading = (sh_basis(F.normalize(normals, dim=-1)) @ light).clamp_min(0) * covered
    lit = dr.antialias((colour * shading).contiguous(), rast, clip, tri)
    alpha = dr.antialias(covered.contiguous(), rast, clip, tri)[..., 0]
    return {
        "colour": lit.clamp(0, 1),
        "albedo": colour,
        "shading": shading,
        "covered": covered[..., 0],
        "alpha": alpha,
    }

## Losses

Per frame: colour (inside the eroded mask, where the head is drawn), outline (above the chin: the
mask also holds the neck and shoulders), the 468 mesh landmarks and the irises. Over the whole
video: smooth motion, sparse expressions, priors, and the size of the shape changes. Held-out
frames see the shared things detached, so only their own pose, face and gaze learn from them.

In [ ]:
def shared_state(detached):
    state = {
        "identity": identity,
        "offsets": offsets(),
        "correctives": correctives(),
        "texture": texture(),
        "wrinkles": wrinkle_maps,
    }
    return {k: v.detach() for k, v in state.items()} if detached else state


def frame_terms(video, frames, shared, light):
    expressions = video.expressions()[frames]
    vertices = posed_vertices(
        shared,
        expressions,
        video.eye_rotations()[frames],
        video.rotations[frames],
        video.translations[frames],
    )
    wrinkles = wrinkles_of(expressions, shared["wrinkles"])
    out = render(vertices, video.camera, shared["texture"], light, wrinkles)
    camera = video.camera

    region = video.inner[frames] * out["covered"]
    error = (out["colour"] - video.images[frames]).abs().mean(-1)
    photo = (error * region).sum() / region.sum().clamp_min(1)

    chin = camera.project(vertices[:, chin_vertex])[:, 1].detach()
    rows = torch.arange(camera.height, device=device).float()
    above = (rows[None, :] < chin[:, None]).float()[:, :, None].expand_as(out["alpha"])
    silhouette = (
        (out["alpha"] - video.masks[frames]).square() * above
    ).sum() / above.sum().clamp_min(1)

    landmarks = keypoint_loss(
        camera.project(vertices[:, dense_vertices]),
        video.landmarks[frames],
        video.landmark_weights[frames],
        camera.height,
    )
    irises = keypoint_loss(
        camera.project(vertices[:, pupils]),
        video.irises[frames],
        video.found[frames][:, None] * facing(vertices, pupils).detach(),
        camera.height,
    )
    return (
        {"photo": photo, "silhouette": silhouette, "landmarks": landmarks, "irises": irises},
        out,
        vertices,
    )


def video_terms(video):
    expressions = video.expressions()
    return {
        "temporal": second_difference(video.rotations).square().sum(-1).mean()
        + second_difference(video.translations).square().sum(-1).mean()
        + 0.01 * expressions.diff(dim=0).square().sum(-1).mean()
        + second_difference(video.eye_angles).square().sum((1, 2)).mean(),
        "sparsity": expressions.sum(-1).mean(),
        "gaze_prior": video.eye_angles.square().sum((1, 2)).mean(),
    }


def shared_terms():
    broad, fine = offset_layers()
    broad_c, fine_c = corrective_layers()
    return {
        "identity_prior": identity.square().sum(),
        "offsets": broad.square().sum(-1).mean(),
        "fine_offsets": fine.square().sum(-1).mean(),
        "correctives": broad_c.square().mean() * 3,  # per vertex and blendshape, like offsets
        "fine_correctives": fine_c.square().mean() * 3,
        "mouth": mouth_shifts.square().sum(-1).mean(),
        "wrinkles": wrinkle_maps.abs().mean(),
    }

## Scores

PSNR on the performer only (inside the eroded mask, where the head is drawn), the 468 landmarks'
mean distance in pixels of the original 1280² frames, and the outline's IoU above the chin. Taken
after phase 1 (the current fit's shape with a learned texture and light: the fair baseline) and
at the end.

In [ ]:
@torch.no_grad()
def scores():
    rows = {}
    shared = shared_state(detached=True)
    for video in videos:
        psnr, landmark_px, iou = [], [], []
        for frames, _ in video.chunks:
            _, out, vertices = frame_terms(video, frames, shared, video.light)
            region = video.inner[frames] * out["covered"]
            mse = ((out["colour"] - video.images[frames]).square().mean(-1) * region).sum(
                (1, 2)
            ) / region.sum((1, 2)).clamp_min(1)
            psnr.append(-10 * torch.log10(mse))
            distance = (
                video.camera.project(vertices[:, dense_vertices]) - video.landmarks[frames]
            ).norm(dim=-1)
            px = distance.mean(-1) / video.scale
            landmark_px.append(torch.where(video.found[frames] > 0, px, torch.nan))
            chin = video.camera.project(vertices[:, chin_vertex])[:, 1]
            rows_ = torch.arange(video.camera.height, device=device).float()
            above = (rows_[None, :] < chin[:, None])[:, :, None]
            drawn, mask = (out["alpha"] > 0.5) & above, (video.masks[frames] > 0.5) & above
            iou.append((drawn & mask).sum((1, 2)) / (drawn | mask).sum((1, 2)).clamp_min(1))
        order = torch.cat([frames for frames, _ in video.chunks]).argsort()
        rows[video.name] = {
            "psnr": torch.cat(psnr)[order].cpu().numpy(),
            "landmark_px": torch.cat(landmark_px)[order].cpu().numpy(),
            "iou": torch.cat(iou)[order].cpu().numpy(),
            "held_out": video.held_out.numpy(),
        }
    return rows


def summary(rows, label):
    for name, r in rows.items():
        for part, chosen in (("trained", ~r["held_out"]), ("held out", r["held_out"])):
            print(
                f"{label:>8} {name:>16} {part:>8}: PSNR {np.nanmean(r['psnr'][chosen]):6.2f} dB, "
                f"landmarks {np.nanmean(r['landmark_px'][chosen]):5.1f} px, "
                f"IoU {np.nanmean(r['iou'][chosen]):.3f}"
            )

## Showing frames

In [ ]:
@torch.no_grad()
def show(picks=((0, 0), (0, 20), (0, 40), (1, 10), (1, 40), (1, 70)), title=""):
    shared = shared_state(detached=True)
    fig, axes = plt.subplots(len(picks), 4, figsize=(13, 3.3 * len(picks)))
    for row, (v, i) in enumerate(picks):
        video = videos[v]
        i = min(i, len(video) - 1)
        frames = torch.tensor([i], device=device)
        _, out, _ = frame_terms(video, frames, shared, video.light)
        image = video.images[i].cpu().numpy()
        render_ = out["colour"][0].cpu().numpy()
        error = np.abs(render_ - image).mean(-1) * out["covered"][0].cpu().numpy()
        for ax, picture, label in zip(
            axes[row],
            (image, render_, out["albedo"][0].clamp(0, 1).cpu().numpy(), error),
            ("frame", "render", "albedo (no light)", "|error|"),
            strict=True,
        ):
            ax.imshow(
                picture,
                cmap="magma" if picture.ndim == 2 else None,
                vmin=0,
                vmax=0.3 if picture.ndim == 2 else None,
            )
            held = " (held out)" if video.held_out[i] else ""
            ax.set_title(f"{video.name} {i}{held}: {label}", fontsize=9)
            ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show(title="the current fit, before any refinement (texture: one mean colour)")

## Optimisation

Each iteration renders every frame (in chunks of `CHUNK`, gradients accumulated), so every
frame's own parameters move every step. One Adam per phase.

In [ ]:
frame_parameters = {
    "pose": [p for v in videos for p in (v.rotations, v.translations)],
    "expressions": [v.expression_logits for v in videos],
    "gaze": [v.eye_angles for v in videos],
}
appearance = ["texture", "light"]
face = appearance + ["pose", "expressions", "gaze", "identity"]
shape = face + ["offsets", "mouth"]
groups = {
    "appearance": appearance,
    "face": face,
    "shape": shape,
    "expressions": shape + ["correctives"],
    "wrinkles": shape + ["correctives", "wrinkles"],
}
parameters_of = {
    "texture": list(pyramid),
    "light": [v.light for v in videos],
    "identity": [identity],
    "offsets": [offset_values, fine_offset_values],
    "mouth": [mouth_shifts],
    "correctives": [corrective_values, fine_corrective_values],
    "wrinkles": [wrinkle_maps],
    **frame_parameters,
}
total_frames = sum(len(v) for v in videos)
history = []


def set_trainable(trainable):
    # parameters a phase does not train get no gradients at all, which saves their backward
    for group, parameters in parameters_of.items():
        for p in parameters:
            p.requires_grad_(group in trainable)


def backward(loss):
    if loss.requires_grad:
        loss.backward()


def iteration_step(trainable):
    # one iteration's gradients; returns the logged terms
    # The shared values (texture pyramid, smoothed shape changes) are built once; every chunk
    # renders from detached copies, whose gradients are pushed back through them at the end.
    shared = shared_state(detached=False)
    leaves = {k: v.detach().requires_grad_(v.requires_grad) for k, v in shared.items()}
    logged = {}
    for video in videos:
        for frames, held in video.chunks:
            used = {k: v.detach() for k, v in leaves.items()} if held else leaves
            light = video.light.detach() if held else video.light
            terms, _, _ = frame_terms(video, frames, used, light)
            share = len(frames) / total_frames
            backward(sum(WEIGHTS[k] * t for k, t in terms.items()) * share)
            for k, t in terms.items():
                logged[k] = logged.get(k, 0.0) + t.item() * share
    pairs = [(shared[k], leaves[k].grad) for k in shared if leaves[k].grad is not None]
    if pairs:
        torch.autograd.backward([value for value, _ in pairs], [grad for _, grad in pairs])
    regular = {}
    for video in videos:
        for k, t in video_terms(video).items():
            regular[k] = regular.get(k, 0.0) + t / len(videos)
    if "identity" in trainable:
        regular.update(shared_terms())
    backward(sum(WEIGHTS[k] * t for k, t in regular.items()))
    logged.update({k: t.item() for k, t in regular.items()})
    return logged


def run_phase(phase):
    trainable = groups[phase]
    set_trainable(trainable)
    optimizer = torch.optim.Adam(
        [{"params": parameters_of[g], "lr": LEARNING_RATES[g]} for g in trainable]
    )
    started = time.time()
    for iteration in range(ITERATIONS[phase]):
        optimizer.zero_grad()
        logged = iteration_step(trainable)
        optimizer.step()
        history.append({"phase": phase, **logged})
        if iteration % 50 == 0 or iteration == ITERATIONS[phase] - 1:
            print(
                f"{phase} {iteration:4d} ({time.time() - started:5.0f} s): "
                + ", ".join(f"{k} {v:.2e}" for k, v in logged.items())
            )

### Profiling (optional)

Times one iteration of every phase (gradients only, no step, nothing learned), then lists the
GPU operations of the slowest one. Set `PROFILE = False` to skip it.

In [ ]:
PROFILE = True


def clear_gradients():
    for parameters in parameters_of.values():
        for p in parameters:
            p.grad = None


if PROFILE:
    timings = {}
    for phase in groups:
        set_trainable(groups[phase])
        iteration_step(groups[phase])  # warm-up: kernels compile and caches fill
        torch.cuda.synchronize()
        started = time.time()
        iteration_step(groups[phase])
        torch.cuda.synchronize()
        timings[phase] = time.time() - started
        clear_gradients()
    for phase, seconds in timings.items():
        minutes = seconds * ITERATIONS[phase] / 60
        print(f"{phase:>12}: {seconds:5.2f} s per iteration, {minutes:5.1f} min for the phase")
    slowest = max(timings, key=timings.get)
    set_trainable(groups[slowest])
    activities = [torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA]
    with torch.profiler.profile(activities=activities) as profiler:
        iteration_step(groups[slowest])
        torch.cuda.synchronize()
    clear_gradients()
    print(f"\nthe slowest phase, {slowest}, by GPU time:")
    print(profiler.key_averages().table(sort_by="cuda_time_total", row_limit=20))

### Phase 1: texture and light on the current fit's geometry

In [ ]:
run_phase("appearance")
before = scores()
summary(before, "before")
show(title="after phase 1: the current fit's shape, learned texture and light")

### Phase 2: + per-frame pose, expressions and gaze, and the identity

In [ ]:
run_phase("face")
summary(scores(), "phase 2")
show(title="after phase 2")

### Phase 3: + smooth shape changes and where the teeth sit

In [ ]:
run_phase("shape")
summary(scores(), "phase 3")
show(title="after phase 3")

### Phase 4: + her own blendshapes

In [ ]:
run_phase("expressions")
summary(scores(), "phase 4")
show(title="after phase 4")

### Phase 5: + wrinkle maps

In [ ]:
run_phase("wrinkles")
after = scores()
summary(before, "before")
summary(after, "after")
show(title="after phase 5")

## Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for ax, key, label in zip(
    axes,
    ("psnr", "landmark_px", "iou"),
    ("PSNR (dB)", "landmarks (px)", "IoU above chin"),
    strict=True,
):
    for name in before:
        ax.plot(before[name][key], "--", label=f"{name} before")
        (line,) = ax.plot(after[name][key], label=f"{name} after")
        held = np.flatnonzero(after[name]["held_out"])
        ax.plot(held, after[name][key][held], "o", color=line.get_color(), markersize=4)
    ax.set_title(label + " per frame (dots: held out)")
    ax.set_xlabel("frame")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 3))
for key in ("photo", "silhouette", "landmarks"):
    ax.semilogy([h.get(key, np.nan) for h in history], label=key)
starts = np.cumsum([0] + [ITERATIONS[p] for p in ITERATIONS])[:-1]
for start, phase in zip(starts, ITERATIONS, strict=True):
    ax.axvline(start, color="grey", lw=0.5)
    ax.text(start, ax.get_ylim()[1], phase, fontsize=8, va="top")
ax.legend()
ax.set_title("loss terms")
plt.show()

The albedo atlas (tiles: face; head and neck; mouth; mouth interior; eyes) and each video's light on a sphere.

In [ ]:
with torch.no_grad():
    atlas = texture()[0].clamp(0, 1).cpu().numpy()
plt.figure(figsize=(18, 4))
plt.imshow(atlas)
plt.axis("off")
plt.show()

side = 128
ys, xs = torch.meshgrid(torch.linspace(1, -1, side), torch.linspace(-1, 1, side), indexing="ij")
inside = xs**2 + ys**2 < 1
# a sphere seen by the fixed camera (world: x right, z up, the camera looks along +y)
n = torch.stack([xs, -(1 - xs**2 - ys**2).clamp_min(0).sqrt(), ys], -1).to(device)
fig, axes = plt.subplots(1, len(videos), figsize=(3 * len(videos), 3))
for ax, video in zip(np.atleast_1d(axes), videos, strict=True):
    with torch.no_grad():
        light = (sh_basis(n) @ video.light).clamp(0, 2) * inside[..., None].to(device)
    ax.imshow((light / light.max()).cpu().numpy())
    ax.set_title(f"{video.name} light")
    ax.axis("off")
plt.show()

The refined neutral head turning, with a uniform light (the albedo as it would sit on the avatar),
beside the same head before the shape changes; and how far the shape moved, in millimetres.

In [ ]:
YAWS = np.linspace(-np.pi, np.pi, 8, endpoint=False)


@torch.no_grad()
def turntable(shared, colour_by=None, yaws=YAWS):
    camera = videos[0].camera
    neutral = torch.zeros(len(yaws), model.num_expressions, device=device)
    eyes = torch.zeros(len(yaws), 2, 3, device=device)
    rotations = torch.zeros(len(yaws), 3, device=device)
    rotations[:, 2] = torch.from_numpy(yaws).float().to(device)
    translation = videos[0].translations.mean(0, keepdim=True).expand(len(yaws), 3)
    vertices = posed_vertices(shared, neutral, eyes, rotations, translation)
    uniform = torch.zeros(9, 3, device=device)
    uniform[0] = 1 / 0.282095
    if colour_by is None:
        out = render(vertices, camera, shared["texture"], uniform)
        return out["colour"].cpu().numpy()
    clip = to_clip(vertices, camera)
    rast, _ = dr.rasterize(context, clip, tri, (camera.height, camera.width))
    value, _ = dr.interpolate(colour_by[None, :, None].contiguous(), rast, tri)
    return (value[..., 0] * (rast[..., 3] > 0)).cpu().numpy()


final = shared_state(detached=True)
unrefined = dict(
    final,
    offsets=torch.zeros_like(final["offsets"]),
    correctives=torch.zeros_like(final["correctives"]),
)
moved = (final["offsets"] / MILLIMETRE).norm(dim=-1)
rows = (
    ("refined", turntable(final)),
    ("without shape changes", turntable(unrefined)),
    ("shape change (mm)", turntable(final, moved)),
)
fig, axes = plt.subplots(3, 8, figsize=(20, 8))
for r, (label, images) in enumerate(rows):
    for c in range(8):
        axes[r, c].imshow(
            images[c],
            cmap="viridis" if images.ndim == 3 else None,
            vmin=0,
            vmax=float(moved.max()) if images.ndim == 3 else None,
        )
        axes[r, c].axis("off")
    axes[r, 0].set_title(label, loc="left")
plt.tight_layout()
plt.show()
print(
    f"shape change: mean {moved[:SKIN_COUNT].mean():.2f} mm, max {moved[:SKIN_COUNT].max():.2f} mm"
)

How much each of her blendshapes differs from ICT-FaceKit's, and the most changed ones side by side.

In [ ]:
with torch.no_grad():
    change = (
        (final["correctives"][:, :SKIN_COUNT] / MILLIMETRE)
        .norm(dim=-1)
        .max(dim=1)
        .values.cpu()
        .numpy()
    )
    used = torch.cat([v.expressions() for v in videos]).max(0).values.cpu().numpy()
order = np.argsort(-change)
fig, ax = plt.subplots(figsize=(16, 3))
ax.bar(range(len(order)), change[order])
ax.set_xticks(
    range(len(order)), [model.expression_names[i] for i in order], rotation=90, fontsize=7
)
ax.set_ylabel("largest change (mm)")
ax2 = ax.twinx()
ax2.plot(range(len(order)), used[order], "k.", label="largest weight in the videos")
ax2.set_ylim(0, 1)
ax2.legend(loc="upper right")
plt.show()


@torch.no_grad()
def expression_image(shared, index):
    camera = videos[0].camera
    weights = torch.zeros(1, model.num_expressions, device=device)
    weights[0, index] = 1
    translation = videos[0].translations.mean(0, keepdim=True)
    vertices = posed_vertices(
        shared,
        weights,
        torch.zeros(1, 2, 3, device=device),
        torch.zeros(1, 3, device=device),
        translation,
    )
    uniform = torch.zeros(9, 3, device=device)
    uniform[0] = 1 / 0.282095
    wrinkles = wrinkles_of(weights, shared["wrinkles"])
    return render(vertices, camera, shared["texture"], uniform, wrinkles)["colour"][0].cpu().numpy()


generic = dict(
    final,
    correctives=torch.zeros_like(final["correctives"]),
    wrinkles=torch.zeros_like(final["wrinkles"]),
)
top = order[:6]
fig, axes = plt.subplots(2, len(top), figsize=(3 * len(top), 6.5))
for c, index in enumerate(top):
    for r, (label, shared) in enumerate((("ICT", generic), ("hers", final))):
        axes[r, c].imshow(expression_image(shared, index))
        axes[r, c].set_title(f"{model.expression_names[index]}: {label}", fontsize=9)
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

The wrinkle maps with the most in them, on the face's tile (grey: no change; lighter and darker:
the colour each blendshape adds at weight 1), and the same blendshapes rendered with and without
them.

In [ ]:
with torch.no_grad():
    strength = final["wrinkles"].abs().mean((1, 2, 3)).cpu().numpy()
strongest = np.argsort(-strength)[:6]
fig, axes = plt.subplots(3, len(strongest), figsize=(3 * len(strongest), 9.5))
no_wrinkles = dict(final, wrinkles=torch.zeros_like(final["wrinkles"]))
for c, index in enumerate(strongest):
    change = final["wrinkles"][index].cpu().numpy()
    axes[0, c].imshow((0.5 + 2 * change).clip(0, 1))
    axes[0, c].set_title(f"{model.expression_names[index]}: {strength[index]:.4f}", fontsize=9)
    axes[1, c].imshow(expression_image(no_wrinkles, index))
    axes[1, c].set_title("without", fontsize=9)
    axes[2, c].imshow(expression_image(final, index))
    axes[2, c].set_title("with wrinkle map", fontsize=9)
    for r in range(3):
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

## Save

Everything the refinement learned, in the head model's own coordinates (metres, ICT axes) and
atlas layout (row 0 of `texture` and of the wrinkle maps is their top; ICT-FaceKit's v is
flipped). `offsets` include the teeth's shifts. The body notebook (`09_body_refinement`) reads it.

In [ ]:
with torch.no_grad():
    record = {
        "identity": identity[0].cpu().numpy(),
        "offsets": final["offsets"].cpu().numpy(),
        "correctives": final["correctives"].cpu().numpy(),
        "expression_names": np.array(model.expression_names),
        "texture": final["texture"][0].cpu().numpy(),
        "wrinkles": final["wrinkles"].cpu().numpy(),
        "mouth_shifts": mouth_shifts.cpu().numpy() * MILLIMETRE,
        "centre": centre.cpu().numpy(),
    }
    for video in videos:
        record[f"{video.name}/light"] = video.light.cpu().numpy()
        record[f"{video.name}/rotations"] = video.rotations.cpu().numpy()
        record[f"{video.name}/translations"] = video.translations.cpu().numpy()
        record[f"{video.name}/expressions"] = video.expressions().cpu().numpy()
        record[f"{video.name}/eye_rotations"] = video.eye_rotations().cpu().numpy()
        for key in ("psnr", "landmark_px", "iou", "held_out"):
            record[f"{video.name}/before_{key}"] = before[video.name][key]
            record[f"{video.name}/after_{key}"] = after[video.name][key]
np.savez_compressed(out_dir / "head.npz", **record)
plt.imsave(out_dir / "albedo.png", atlas)
print("written to", out_dir)